In [ ]:
# imports

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import ollama

In [ ]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [ ]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

openai = OpenAI()

In [ ]:
# here is the question; type over this to ask something new

question = """
What is the bias-variance tradeoff, and how does it show up differently in classical ML models versus overparameterized deep neural networks?"
"""

In [ ]:
# system prompt: sets the model's role, audience and teaching style for every answer

system_prompt = """
You are an expert Machine Learning mentor and tutor. You are helping a student who is:
- Currently completing a Master's degree in Computer Science, specializing in Machine Learning.
- Preparing to apply for a PhD in Machine Learning at top universities in Europe.
- Working toward a long-term career goal of becoming a Research Scientist / Engineer at Google.

Your job is to answer technical questions (about code, ML/AI concepts, math, tools or career topics) in a way
that builds this student's expertise for that exact path. Follow these rules every time you answer:

1. Explain every detail simply and clearly, as if teaching someone who wants deep understanding, not just a
   quick answer. Do not skip a step because it seems "obvious".
2. If the question involves code, walk through it piece by piece (e.g. syntax, data structures, control flow)
   before giving the overall summary.
3. Define any technical jargon the first time you use it, in one short plain-English sentence.
4. Where relevant, connect the explanation to underlying ML/CS fundamentals (algorithms, statistics,
   optimization, deep learning theory) since that depth matters for PhD-level research.
5. Where relevant, briefly mention how the concept could appear in a research paper, PhD coursework, or a
   technical interview at a company like Google, so the student sees why it matters for their goals.
6. Use Markdown: headers, bullet points, code blocks, and bold key terms so the explanation is easy to scan.
7. Keep a warm, encouraging, mentor-like tone - this student is working hard toward an ambitious goal.
"""

In [ ]:
# user prompt: carries my personal context plus the actual question, so the model always knows who's asking

user_prompt = f"""
I'm a Computer Science Master's student specializing in Machine Learning. I'm preparing to apply for a PhD in
ML at top European universities, and my long-term goal is to work as a Research Scientist at Google.

Please explain the following in a way that gives me deep, fundamental understanding (not just a surface-level
answer), breaking down every detail simply, and connecting it to concepts that matter for PhD research and
technical interviews where relevant:

{question}
"""

# messages: combines the system prompt (how to answer) and user prompt (what is being asked)
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

In [ ]:
# Get gpt-4o-mini to answer, with streaming

stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=messages,
    stream=True
)

response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    response = response.replace("```markdown", "").replace("```", "")
    update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
# Get Llama 3.2 to answer

response = ollama.chat(model=MODEL_LLAMA, messages=messages)
reply = response['message']['content']
display(Markdown(reply))